# Real Estate Pricing Analysis

Exploring price per square foot across all properties and focusing on Itajaí.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Load Dataset

In [ ]:
df = pd.read_csv('properties_rows.csv', low_memory=False)
df.head()

## Data Cleaning

In [ ]:
df['price_usd'] = pd.to_numeric(df['price_usd'], errors='coerce')
df['area_sqft'] = pd.to_numeric(df['area_sqft'], errors='coerce')
df['views_count'] = pd.to_numeric(df['views_count'], errors='coerce')

df = df.dropna(subset=['price_usd', 'area_sqft'])
df = df[(df['price_usd'] > 0) & (df['area_sqft'] > 0)]

## Feature Engineering

In [ ]:
df['price_per_sqft'] = df['price_usd'] / df['area_sqft']

## Distribution (All Data)

In [ ]:
plt.figure()
plt.hist(df['price_per_sqft'], bins=30)
plt.title('Distribution of Price per Sqft (All Data)')
plt.xlabel('Price per Sqft')
plt.ylabel('Frequency')
plt.show()

# Distribution of Price per Sqft by City

In [ ]:
# Select top cities by number of listings
top_cities = df['city'].value_counts().head(5).index

# Filter dataset
df_top = df[df['city'].isin(top_cities)].copy()

# Boxplot
plt.figure()
df_top.boxplot(column='price_per_sqft', by='city')

plt.title('Price per Sqft Distribution by City (Top 5)')
plt.suptitle('')  # remove default title
plt.xlabel('City')
plt.ylabel('Price per Sqft')

plt.show()

## Filter Itajaí

In [ ]:
df_itajai = df[df['city'].astype(str).str.lower().str.contains('itaj', na=False)].copy()

print(f"Number of properties in Itajaí: {len(df_itajai)}")

## Distribution (Itajaí)

In [ ]:
plt.figure()
plt.hist(df_itajai['price_per_sqft'], bins=30)
plt.title('Distribution of Price per Sqft (Itajaí)')
plt.xlabel('Price per Sqft')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_itajai = df[df['city'].astype(str).str.lower().str.contains('itaj', na=False)].copy()

print(f"Total Itajaí: {len(df_itajai)}")

In [ ]:
top_expensive_itajai = df_itajai.sort_values(by='price_per_sqft', ascending=False).head(10)

top_expensive_itajai[['price_usd', 'area_sqft', 'price_per_sqft']]


In [ ]:
top_cheap_itajai = df_itajai.sort_values(by='price_per_sqft', ascending=True).head(10)

top_cheap_itajai[['price_usd', 'area_sqft', 'price_per_sqft']]

In [ ]:
# ============================================
# Price per Sqft vs Area (Itajaí)
# ============================================

import matplotlib.pyplot as plt

# Filter Itajaí (robust - handles Itajai / Itajaí)
df_itajai = df[df['city'].astype(str).str.lower().str.contains('itaj', na=False)].copy()

# Plot scatter
plt.figure()
plt.scatter(df_itajai['area_sqft'], df_itajai['price_per_sqft'])

plt.title('Price per Sqft vs Area (Itajaí)')
plt.xlabel('Area (sqft)')
plt.ylabel('Price per Sqft')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Approximate beach location (Itajaí)
BEACH_LAT = -26.9101
BEACH_LON = -48.6343

# Haversine function (distance in km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

# Calculate distance to beach
df_itajai['distance_to_beach_km'] = haversine(
    df_itajai['latitude'],
    df_itajai['longitude'],
    BEACH_LAT,
    BEACH_LON
)

# ============================================
# Plot: Distribution of distance to beach
# ============================================

plt.figure()
plt.hist(df_itajai['distance_to_beach_km'], bins=30)

plt.title('Distribution of Distance to Beach (Itajaí)')
plt.xlabel('Distance to Beach (km)')
plt.ylabel('Frequency')

plt.show()

In [ ]:
# Distance vs price_per_sqft

plt.figure()
plt.scatter(df_itajai['distance_to_beach_km'], df_itajai['price_per_sqft'])

plt.title('Distance to Beach vs Price per Sqft (Itajaí)')
plt.xlabel('Distance to Beach (km)')
plt.ylabel('Price per Sqft')

plt.show()

In [ ]:
# Approximate points along Praia Brava coastline
beach_points = [
    (-26.9450, -48.6170),
    (-26.9500, -48.6150),
    (-26.9550, -48.6130),
    (-26.9600, -48.6110),
    (-26.9650, -48.6090),
]

In [ ]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km

    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [ ]:
def min_distance_to_beach(row, beach_points):
    distances = [
        haversine(row['latitude'], row['longitude'], lat, lon)
        for lat, lon in beach_points
    ]
    return min(distances)

df_itajai['distance_to_beach_km'] = df_itajai.apply(
    lambda row: min_distance_to_beach(row, beach_points),
    axis=1
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.scatter(df_itajai['distance_to_beach_km'], df_itajai['price_per_sqft'])

plt.title('Distance to Praia Brava vs Price per Sqft (Itajaí)')
plt.xlabel('Distance to Beach (km)')
plt.ylabel('Price per Sqft')

plt.show()